# SortSense — Colab Demo (Capstone Q3)

Runnable link that reproduces **real inference** on the trained YOLOv8 waste detector, and (optionally) **reproduces training from scratch**. Everything is self-contained: this notebook clones the public GitHub repo and unzips the bundled dataset — no manual uploads needed.

Repo: https://github.com/nutapol-project/-computer-vision

Runtime → Change runtime type → T4 GPU (optional for the inference section, recommended if you run the training section).

In [ ]:
!pip -q install ultralytics==8.4.160
import torch, os
print('GPU available:', torch.cuda.is_available())
if not os.path.isdir('/content/repo'):
    !git clone --depth 1 https://github.com/nutapol-project/-computer-vision.git /content/repo
%cd /content/repo

## 1. Unzip the bundled dataset
Same dataset described in the report: 200 images / 229 boxes, Train 151 / Val 25 / Test 24.

In [ ]:
import zipfile, pathlib
DATA_DIR = pathlib.Path('/content/repo/dataset_waste_real_v3_hazard_single')
if not any(DATA_DIR.glob('images/test/*.jpg')):
    with zipfile.ZipFile('/content/repo/data/dataset_waste_real_v3_hazard_single.zip') as z:
        z.extractall(DATA_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()))
print('test images:', len(list((DATA_DIR / "images/test").glob('*.jpg'))))

## 2. Reproduce inference on real test images
Loads the actual trained weights (`best.pt`) and runs them on two real, held-out test images — the same success case and honest failure case shown in the demo video and report:
- `1_13.jpg`: ground truth is a single Recyclable object.
- `H93.jpg`: ground truth has **two** separate Hazardous objects (a broken phone screen and an exposed battery). The model has historically returned only **one** merged box here — that's the honest failure case discussed in the report (Hazardous recall ~50%).

In [ ]:
from ultralytics import YOLO
from PIL import Image

model = YOLO('/content/repo/best.pt')
print('classes:', model.names)

images = [DATA_DIR / 'images/test/1_13.jpg', DATA_DIR / 'images/test/H93.jpg']
results = model.predict(source=[str(p) for p in images], conf=0.45, imgsz=640, save=False)
for img_path, result in zip(images, results):
    preds = [(model.names[int(b.cls.item())], round(float(b.conf.item()), 3)) for b in result.boxes]
    print(img_path.name, '-> detections:', preds)
    display(Image.fromarray(result.plot()[..., ::-1]))

## 3. (Optional) Reproduce training from scratch
Retrains YOLOv8n (the baseline) on the same split, same seed (42), and the same hyperparameters documented in the report and README ("Baseline comparison" section): epochs ≤ 100, patience 25, batch 8, image size 640. Takes a few minutes on a free T4 GPU — slow on CPU, so enable the GPU runtime first if you run this cell.

In [ ]:
import yaml

cfg = yaml.safe_load((DATA_DIR / 'data.yaml').read_text())
cfg['path'] = str(DATA_DIR.resolve())
colab_yaml = pathlib.Path('/content/colab-data.yaml')
colab_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(colab_yaml.read_text())

train_model = YOLO('yolov8n.pt')
train_model.train(data=str(colab_yaml), epochs=100, imgsz=640, batch=8,
                   seed=42, patience=25, device=0 if torch.cuda.is_available() else 'cpu',
                   project='/content/runs', name='colab_baseline', plots=True)

In [ ]:
best_run = YOLO('/content/runs/colab_baseline/weights/best.pt')
metrics = best_run.val(data=str(colab_yaml), split='test',
                        device=0 if torch.cuda.is_available() else 'cpu')
print({k: round(float(v), 4) for k, v in metrics.results_dict.items()})

## Links
- Full report (PDF/Word), architecture diagram, and results: `report/` in the repo
- Baseline-comparison notebooks (Kaggle format): `kaggle_train_waste_real_v3_hazard_single.ipynb` (YOLOv8n) and `kaggle_train_waste_real_v3_yolov8m.ipynb` (YOLOv8m)
- `compare_models.py` turns two `test-metrics.json` outputs into the baseline-vs-main-model table